# LeetCode 2614 — Prime In Diagonal bằng Python

Link đề: https://leetcode.com/problems/prime-in-diagonal/ · Độ khó: **Easy** · Chủ đề: Array, Math, Matrix, Number Theory

Skeleton LeetCode:

```python
class Solution:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biết **đường chéo chính** và **đường chéo phụ** của ma trận vuông được viết bằng chỉ số như thế nào: `nums[i][i]` và `nums[i][n - 1 - i]`.
2. Hiểu **số nguyên tố** là gì và viết được hàm kiểm tra bằng **chia thử đến căn bậc hai** (trial division), kể cả các bẫy như số `1`, số `2`, số chính phương.
3. Biết ba kỹ thuật bổ trợ: `set` để loại giá trị trùng, `sorted(..., reverse=True)` để duyệt từ lớn đến nhỏ rồi **dừng sớm**, và **sàng Eratosthenes** (sieve) để lập bảng số nguyên tố một lần.
4. Viết 4 cách: chia thử đến `v - 1` (chậm), chia thử đến `√v`, "cắt tỉa" bằng biến `best` / sắp xếp giảm dần, và sàng Eratosthenes; rồi đối chiếu chúng bằng test ngẫu nhiên.
5. Tránh các lỗi kinh điển: coi `1` là số nguyên tố, quên ô trùng ở tâm ma trận lẻ, dùng sai chỉ số đường chéo phụ, vòng lặp chia thử bỏ sót căn nguyên.

## 2. Đề bài & ví dụ

Cho ma trận vuông `nums` (mảng hai chiều, đánh chỉ số từ 0). Hãy trả về **số nguyên tố lớn nhất nằm trên ít nhất một đường chéo** của `nums`. Nếu trên hai đường chéo không có số nguyên tố nào thì trả về `0`.

Lưu ý:

- Một số nguyên là **số nguyên tố** nếu nó **lớn hơn 1** và chỉ có đúng hai ước dương là `1` và chính nó.
- Một giá trị `val` "nằm trên đường chéo" nếu tồn tại `i` sao cho `nums[i][i] == val` **hoặc** `nums[i][nums.length - i - 1] == val`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[[1,2,3],[5,6,7],[9,10,11]]` | `11` | Các số trên đường chéo: `1, 3, 6, 9, 11`; số nguyên tố lớn nhất là `11`. |
| 2 | `[[1,2,3],[5,17,7],[9,11,10]]` | `17` | Các số trên đường chéo: `1, 3, 9, 10, 17`; trong đó nguyên tố là `3` và `17`, lớn nhất là `17`. (Số `7` và `11` không nằm trên đường chéo.) |

Constraints:

- `1 <= nums.length <= 300`
- `nums.length == nums[i].length` (ma trận vuông)
- `1 <= nums[i][j] <= 4 * 10^6`

Chú ý: ma trận chỉ có tối đa `2 * 300 - 1 = 599` ô nằm trên đường chéo, nhưng mỗi giá trị có thể lên tới bốn triệu — nên **cách kiểm tra nguyên tố** mới là chỗ quyết định tốc độ.

In [1]:
# Chạy thử dữ liệu của ví dụ 1: in ra từng ô nằm trên hai đường chéo
nums = [[1, 2, 3], [5, 6, 7], [9, 10, 11]]
n = len(nums)
for i in range(n):
    print(f"i={i}: chéo chính nums[{i}][{i}] = {nums[i][i]}, chéo phụ nums[{i}][{n - 1 - i}] = {nums[i][n - 1 - i]}")

i=0: chéo chính nums[0][0] = 1, chéo phụ nums[0][2] = 3
i=1: chéo chính nums[1][1] = 6, chéo phụ nums[1][1] = 6
i=2: chéo chính nums[2][2] = 11, chéo phụ nums[2][0] = 9


## 3. Kiến thức nền

### 3.1. Đường chéo chính và đường chéo phụ

Ma trận `nums` có `n` hàng, `n` cột. Ô ở hàng `i`, cột `j` là `nums[i][j]`.

- **Đường chéo chính** đi từ góc trên-trái xuống góc dưới-phải: các ô có **hàng = cột**, tức `nums[i][i]`.
- **Đường chéo phụ** đi từ góc trên-phải xuống góc dưới-trái: các ô có **hàng + cột = n - 1**, tức cột `j = n - 1 - i`, ô `nums[i][n - 1 - i]`.

Với `n = 3`: chéo chính là `(0,0), (1,1), (2,2)`; chéo phụ là `(0,2), (1,1), (2,0)`. Ô chính giữa `(1,1)` nằm trên **cả hai** đường chéo (khi `n` lẻ).

In [2]:
n = 4
for i in range(n):
    print(f"hàng {i}: chéo chính cột {i}, chéo phụ cột {n - 1 - i}")
# n chẵn thì hai đường chéo KHÔNG cắt nhau; n lẻ thì cắt nhau ở ô tâm (i == n - 1 - i)
print([i for i in range(n) if i == n - 1 - i])        # [] vì n = 4 chẵn
print([i for i in range(5) if i == 5 - 1 - i])        # [2] vì n = 5 lẻ

hàng 0: chéo chính cột 0, chéo phụ cột 3
hàng 1: chéo chính cột 1, chéo phụ cột 2
hàng 2: chéo chính cột 2, chéo phụ cột 1
hàng 3: chéo chính cột 3, chéo phụ cột 0
[]
[2]


### 3.2. Số nguyên tố và ước số

Số nguyên tố: `2, 3, 5, 7, 11, 13, ...`. **Không phải** số nguyên tố: `1` (chỉ có một ước), `4 = 2 × 2`, `9 = 3 × 3`, `15 = 3 × 5`.

Để kiểm tra `v` có phải số nguyên tố không, ta tìm xem có số `d` nào với `2 <= d < v` mà `v % d == 0` hay không. Phép `%` cho **số dư**: `15 % 3 == 0` (chia hết), `15 % 4 == 3` (dư 3).

In [3]:
v = 15
print([d for d in range(2, v) if v % d == 0])   # các ước thật sự của 15: [3, 5]
v = 13
print([d for d in range(2, v) if v % d == 0])   # [] -> không có ước nào -> 13 là nguyên tố

[3, 5]
[]


### 3.3. Vì sao chỉ cần thử đến căn bậc hai?

Nếu `v = a × b` với `a <= b` thì `a × a <= a × b = v`, nên `a <= √v`. Nghĩa là: **mọi hợp số đều có một ước nhỏ nhất không vượt quá `√v`**. Vậy nếu đã thử hết `d` từ `2` đến `√v` mà không có ước nào thì chắc chắn `v` là nguyên tố.

Ví dụ `v = 36`: các cặp ước `(2,18), (3,12), (4,9), (6,6)`. Ước nhỏ hơn của mỗi cặp đều `<= 6 = √36`.

Để tránh số thực, ta dùng điều kiện `d * d <= v` (số nguyên hoàn toàn) thay cho `d <= sqrt(v)`. Có thể dùng `math.isqrt(v)` — căn bậc hai **nguyên** (làm tròn xuống) — và vòng `range(2, isqrt(v) + 1)`.

In [4]:
import math
for v in [36, 37, 49, 50]:
    r = math.isqrt(v)
    print(f"v={v}: isqrt={r}, thử d trong range(2, {r + 1}) = {list(range(2, r + 1))}")
print(math.isqrt(48), math.isqrt(49))   # 6 7 : làm tròn xuống, 49 = 7*7 chính xác

v=36: isqrt=6, thử d trong range(2, 7) = [2, 3, 4, 5, 6]
v=37: isqrt=6, thử d trong range(2, 7) = [2, 3, 4, 5, 6]
v=49: isqrt=7, thử d trong range(2, 8) = [2, 3, 4, 5, 6, 7]
v=50: isqrt=7, thử d trong range(2, 8) = [2, 3, 4, 5, 6, 7]
6 7


### 3.4. `set`, `sorted(..., reverse=True)` và `any` / `all`

- `set` là tập hợp không có phần tử trùng: `set([3, 17, 3])` được `{3, 17}`. Giúp ta chỉ kiểm tra mỗi giá trị **một lần**.
- `sorted(s, reverse=True)` trả về danh sách **giảm dần**: giá trị lớn nhất đứng đầu.
- `all(điều kiện for d in ...)` đúng khi **mọi** phần tử đều thỏa; `any(...)` đúng khi **có ít nhất một** phần tử thỏa. Cả hai dừng ngay khi đã biết đáp án (đoản mạch).

In [5]:
vals = [1, 3, 9, 10, 17, 3, 17]
s = set(vals)
print(s)                              # {1, 3, 9, 10, 17}
print(sorted(s, reverse=True))        # [17, 10, 9, 3, 1]
print(all(15 % d != 0 for d in range(2, 4)))   # False vì 15 % 3 == 0
print(any(15 % d == 0 for d in range(2, 4)))   # True
print(all(x > 0 for x in []))         # True: tập rỗng thì "mọi" luôn đúng

{1, 3, 9, 10, 17}
[17, 10, 9, 3, 1]
False
True
True


### 3.5. Tạo mảng đánh dấu với `bytearray`

Ta sẽ cần "bảng" `is_p[0..M]` ghi lại số nào còn là nguyên tố. `bytearray(M + 1)` là mảng các byte `0`; `bytearray([1]) * (M + 1)` là mảng toàn `1`. Gán lát cắt theo bước: `a[4::2] = bytes(len(a[4::2]))` đặt `0` cho `4, 6, 8, ...`. Dùng `bytearray` tiết kiệm bộ nhớ (1 byte mỗi phần tử) so với list Python.

In [6]:
M = 20
a = bytearray([1]) * (M + 1)
a[0] = a[1] = 0
a[4::2] = bytes(len(a[4::2]))          # xóa các bội của 2 bắt đầu từ 4
print(list(a))
print([i for i in range(M + 1) if a[i]])   # còn 2,3,5,7,9,11,13,15,17,19 (chưa loại bội của 3, 5...)

[0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
[2, 3, 5, 7, 9, 11, 13, 15, 17, 19]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Bài toán gồm **hai việc độc lập**:

1. **Thu thập** các giá trị nằm trên hai đường chéo (tối đa `2n - 1` giá trị).
2. **Lọc** các giá trị nguyên tố rồi lấy **lớn nhất** (mặc định `0`).

Tính tay ví dụ 2: `[[1,2,3],[5,17,7],[9,11,10]]`.

- Chéo chính: `1, 17, 10`. Chéo phụ: `3, 17, 9`.
- Gộp lại: `1, 17, 10, 3, 17, 9`.
- `1` không nguyên tố; `17` nguyên tố; `10 = 2×5` không; `3` nguyên tố; `9 = 3×3` không.
- Nguyên tố còn lại: `17, 3` ⇒ lớn nhất `17`.

Như vậy toàn bộ độ khó nằm ở hàm **"`v` có phải số nguyên tố không?"**, và ở việc gọi nó **ít lần nhất có thể**.

In [7]:
nums = [[1, 2, 3], [5, 17, 7], [9, 11, 10]]
n = len(nums)
vals = [nums[i][i] for i in range(n)] + [nums[i][n - 1 - i] for i in range(n)]
print("giá trị trên đường chéo:", vals)
print("không trùng:", sorted(set(vals)))

giá trị trên đường chéo: [1, 17, 10, 3, 17, 9]
không trùng: [1, 3, 9, 10, 17]


## 5. Cách 1 — Brute Force / baseline: chia thử mọi `d` từ `2` đến `v - 1`

**Ý tưởng:** duyệt mọi ô trên hai đường chéo; với mỗi giá trị `v` dùng định nghĩa nguyên tố trực tiếp: `v > 1` và không có `d` nào trong `2 .. v - 1` chia hết `v`. Giữ lại giá trị nguyên tố lớn nhất.

Cách này **đúng** nhưng chậm: kiểm tra một số cỡ `4 × 10^6` có thể tốn tới 4 triệu phép chia.

In [8]:
from typing import List

def is_prime_slow(v: int) -> bool:
    if v < 2:
        return False
    for d in range(2, v):          # thử TẤT CẢ d từ 2 đến v - 1
        if v % d == 0:
            return False
    return True

class SolutionBrute:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        n = len(nums)
        best = 0
        for i in range(n):
            for v in (nums[i][i], nums[i][n - 1 - i]):
                if is_prime_slow(v) and v > best:
                    best = v
        return best

In [9]:
sol = SolutionBrute()
print(sol.diagonalPrime([[1, 2, 3], [5, 6, 7], [9, 10, 11]]))   # 11
print(sol.diagonalPrime([[1, 2, 3], [5, 17, 7], [9, 11, 10]]))  # 17
print(sol.diagonalPrime([[1]]))                                 # 0 (1 không là nguyên tố)

11
17
0


**Độ phức tạp:** gọi `n` là cạnh ma trận, `M` là giá trị lớn nhất. Có `O(n)` ô trên đường chéo, mỗi ô tốn tối đa `O(M)` phép chia ⇒ `O(n · M)` — với `n = 300`, `M = 4×10^6` là cỡ hàng tỷ, **quá chậm**. Space `O(1)`.

**Nút thắt:** ta kiểm tra nguyên tố quá lãng phí. Nếu `v` là hợp số thì ước nhỏ nhất của nó rất nhỏ; còn nếu `v` là nguyên tố ta vẫn phải thử tới tận `v - 1` dù ước (nếu có) chỉ nằm ở nửa đầu.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — chỉ thử đến `√v`.** Như mục 3.3: nếu `v` có ước thật sự thì có ước `<= √v`. Với `v <= 4×10^6` ta chỉ thử tối đa `2000` giá trị `d` (vì `√(4×10^6) = 2000`). Giảm từ `O(v)` xuống `O(√v)` mỗi lần kiểm tra.

**Đột phá 2 — đừng kiểm tra những giá trị không cần.** Ta chỉ quan tâm số nguyên tố **lớn nhất**. Hai cách tránh việc thừa:

- Giữ biến `best`; chỉ kiểm tra `v` khi `v > best` (giá trị nhỏ hơn không thể cải thiện kết quả).
- Hoặc gom giá trị vào `set`, sắp xếp **giảm dần**, rồi kiểm tra từ lớn xuống nhỏ — số nguyên tố đầu tiên tìm được **chính là** đáp án, dừng ngay.

**Đột phá 3 — sàng Eratosthenes.** Nếu muốn kiểm tra nhiều số cùng lúc, ta lập sẵn bảng nguyên tố đến `M = max` bằng cách gạch bội của từng số nguyên tố; sau đó mỗi lần kiểm tra chỉ là một lần tra bảng `O(1)`.

In [10]:
import math
for v in [1_000_003, 4_000_000]:
    print(v, "=> số phép chia tối đa:", "v-1 =", v - 1, "| căn:", math.isqrt(v))

1000003 => số phép chia tối đa: v-1 = 1000002 | căn: 1000
4000000 => số phép chia tối đa: v-1 = 3999999 | căn: 2000


## 7. Các cách tiếp theo

### Cách 2 — Chia thử đến `√v`

Hàm kiểm tra nguyên tố tốt hơn: xử lý `v < 2` → `False`, rồi thử `d = 2, 3, ..., isqrt(v)`. Các đoạn:

- `if v < 2: return False` — loại `1` (đề cho `nums[i][j] >= 1`, nên `1` chắc chắn xuất hiện).
- `for d in range(2, isqrt(v) + 1)` — cận trên **bao gồm** căn nguyên (ví dụ `v = 9` cần thử `d = 3`).
- `if v % d == 0: return False` — tìm thấy ước thật sự ⇒ hợp số.
- Hết vòng lặp không thấy ước ⇒ `True`.

In [11]:
import math

def is_prime(v: int) -> bool:
    if v < 2:
        return False
    for d in range(2, math.isqrt(v) + 1):
        if v % d == 0:
            return False
    return True

class SolutionSqrt:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        n = len(nums)
        best = 0
        for i in range(n):
            for v in (nums[i][i], nums[i][n - 1 - i]):
                if is_prime(v) and v > best:
                    best = v
        return best

print([x for x in range(1, 30) if is_prime(x)])
print(SolutionSqrt().diagonalPrime([[1, 2, 3], [5, 17, 7], [9, 11, 10]]))   # 17

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
17


**Độ phức tạp:** mỗi lần kiểm tra `O(√M)`; có `O(n)` ô ⇒ Time `O(n · √M)`. Với số liệu tối đa: `599 × 2000 ≈ 1.2 × 10^6` phép chia — rất nhanh. Space `O(1)`.

### Cách 3 — Cắt tỉa theo `best` (bỏ qua giá trị không thể cải thiện)

Giữ `best` = số nguyên tố lớn nhất đã tìm. Với mỗi ô, nếu `v <= best` thì **không cần kiểm tra** (dù nguyên tố cũng không làm đáp án tốt hơn). Hàm `is_prime` chỉ được gọi khi thực sự cần: câu lệnh `if v > best and is_prime(v)` tận dụng **đoản mạch** của `and`. Đây là cách ta sẽ nộp.

In [12]:
class SolutionPrune:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        n = len(nums)
        best = 0
        for i in range(n):
            for v in (nums[i][i], nums[i][n - 1 - i]):
                if v > best and is_prime(v):     # v <= best thì bỏ qua, không gọi is_prime
                    best = v
        return best

print(SolutionPrune().diagonalPrime([[1, 2, 3], [5, 6, 7], [9, 10, 11]]))   # 11

11


**Độ phức tạp:** Time tệ nhất vẫn `O(n · √M)` (khi các giá trị tăng dần, mỗi giá trị đều phải kiểm tra), nhưng thực tế ít lần gọi `is_prime` hơn. Space `O(1)`.

### Cách 4 — Sắp xếp giảm dần + dừng sớm

Gom giá trị đường chéo vào `set` (bỏ trùng, tâm ma trận lẻ chỉ tính một lần), sắp xếp **giảm dần**, duyệt từ lớn đến nhỏ; số nguyên tố đầu tiên gặp là lớn nhất ⇒ `return` ngay. Không có thì trả `0`.

In [13]:
class SolutionSortedEarly:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        n = len(nums)
        vals = set()
        for i in range(n):
            vals.add(nums[i][i])
            vals.add(nums[i][n - 1 - i])
        for v in sorted(vals, reverse=True):
            if is_prime(v):
                return v           # lớn nhất vì đang duyệt giảm dần
        return 0

print(SolutionSortedEarly().diagonalPrime([[1, 2, 3], [5, 17, 7], [9, 11, 10]]))   # 17
print(SolutionSortedEarly().diagonalPrime([[1]]))                                    # 0

17
0


**Độ phức tạp:** gom + sắp xếp `O(n log n)` (có tối đa `2n - 1` giá trị); số lần gọi `is_prime` thường rất ít (dừng ở số nguyên tố đầu tiên), tệ nhất `O(n · √M)`. Space `O(n)` cho `set`.

### Cách 5 — Sàng Eratosthenes đến `max` của đường chéo

Ý tưởng: tính `M = max(giá trị đường chéo)`. Lập bảng `is_p[0..M]`; với mỗi `p` từ `2` đến `√M` còn nguyên tố, gạch các bội `p*p, p*p + p, ...` Sau đó duyệt đường chéo và tra bảng.

Vì sao bắt đầu gạch từ `p * p`? Các bội nhỏ hơn như `2p, 3p, ...` đã bị gạch bởi số nguyên tố nhỏ hơn rồi.

In [14]:
class SolutionSieve:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        n = len(nums)
        diag = [nums[i][i] for i in range(n)] + [nums[i][n - 1 - i] for i in range(n)]
        M = max(diag)
        is_p = bytearray([1]) * (M + 1)
        is_p[0] = 0
        if M >= 1:
            is_p[1] = 0
        for p in range(2, math.isqrt(M) + 1):
            if is_p[p]:
                is_p[p * p::p] = bytes(len(is_p[p * p::p]))
        best = 0
        for v in diag:
            if is_p[v] and v > best:
                best = v
        return best

print(SolutionSieve().diagonalPrime([[1, 2, 3], [5, 17, 7], [9, 11, 10]]))   # 17
print(SolutionSieve().diagonalPrime([[4, 6], [8, 9]]))                         # 0

17
0


**Độ phức tạp:** Time `O(M log log M)` để sàng + `O(n)` để tra; Space `O(M)` (với `M = 4×10^6` là 4 MB, chấp nhận được). Sàng hợp lý khi có **rất nhiều** giá trị cần kiểm tra; với tối đa 599 giá trị thì chia thử đến `√v` vừa gọn vừa không kém.

## 8. TRACE từng bước

Trace Cách 3 (cắt tỉa theo `best`) trên `[[1,2,3],[5,17,7],[9,11,10]]`:

| `i` | `v` (ô) | `v > best`? | `is_prime(v)`? | `best` sau |
|---|---|---|---|---|
| 0 | `nums[0][0] = 1` | `1 > 0` có | không (`1 < 2`) | `0` |
| 0 | `nums[0][2] = 3` | `3 > 0` có | có | `3` |
| 1 | `nums[1][1] = 17` | `17 > 3` có | có | `17` |
| 1 | `nums[1][1] = 17` (ô tâm lần 2) | `17 > 17` không | **bỏ qua** | `17` |
| 2 | `nums[2][2] = 10` | `10 > 17` không | bỏ qua | `17` |
| 2 | `nums[2][0] = 9` | `9 > 17` không | bỏ qua | `17` |

Kết quả `17`.

In [15]:
def trace_prune(nums):
    n = len(nums)
    best = 0
    for i in range(n):
        for name, j in (("chéo chính", i), ("chéo phụ", n - 1 - i)):
            v = nums[i][j]
            if v > best:
                ok = is_prime(v)
                print(f"i={i} {name:10} nums[{i}][{j}]={v:3} > best={best:3} -> is_prime={ok}")
                if ok:
                    best = v
            else:
                print(f"i={i} {name:10} nums[{i}][{j}]={v:3} <= best={best:3} -> bỏ qua")
    print("kết quả:", best)
    return best

trace_prune([[1, 2, 3], [5, 17, 7], [9, 11, 10]])

i=0 chéo chính nums[0][0]=  1 > best=  0 -> is_prime=False
i=0 chéo phụ   nums[0][2]=  3 > best=  0 -> is_prime=True
i=1 chéo chính nums[1][1]= 17 > best=  3 -> is_prime=True
i=1 chéo phụ   nums[1][1]= 17 <= best= 17 -> bỏ qua
i=2 chéo chính nums[2][2]= 10 <= best= 17 -> bỏ qua
i=2 chéo phụ   nums[2][0]=  9 <= best= 17 -> bỏ qua
kết quả: 17


17

In [16]:
def trace_is_prime(v):
    print(f"kiểm tra v={v}, isqrt={math.isqrt(v)}")
    if v < 2:
        print("  v < 2 => không nguyên tố")
        return False
    for d in range(2, math.isqrt(v) + 1):
        print(f"  d={d}: {v} % {d} = {v % d}")
        if v % d == 0:
            print(f"  => {v} = {d} x {v // d}, hợp số")
            return False
    print("  không có ước nào => nguyên tố")
    return True

trace_is_prime(49)
trace_is_prime(53)
trace_is_prime(2)

kiểm tra v=49, isqrt=7
  d=2: 49 % 2 = 1
  d=3: 49 % 3 = 1
  d=4: 49 % 4 = 1
  d=5: 49 % 5 = 4
  d=6: 49 % 6 = 1
  d=7: 49 % 7 = 0
  => 49 = 7 x 7, hợp số
kiểm tra v=53, isqrt=7
  d=2: 53 % 2 = 1
  d=3: 53 % 3 = 2
  d=4: 53 % 4 = 1
  d=5: 53 % 5 = 3
  d=6: 53 % 6 = 5
  d=7: 53 % 7 = 4
  không có ước nào => nguyên tố
kiểm tra v=2, isqrt=1
  không có ước nào => nguyên tố


True

## 9. Code nộp LeetCode

Dùng Cách 3 (cắt tỉa theo `best`) cùng hàm kiểm tra nguyên tố chia thử đến `√v`. Trên LeetCode ta đặt hàm phụ vào trong class (static method hoặc hàm lồng) để file nộp gói gọn.

In [17]:
import math
from typing import List

class Solution:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        def is_prime(v: int) -> bool:
            if v < 2:
                return False
            for d in range(2, math.isqrt(v) + 1):
                if v % d == 0:
                    return False
            return True

        n = len(nums)
        best = 0
        for i in range(n):
            for v in (nums[i][i], nums[i][n - 1 - i]):
                if v > best and is_prime(v):
                    best = v
        return best

In [18]:
print(Solution().diagonalPrime([[1, 2, 3], [5, 6, 7], [9, 10, 11]]))   # 11
print(Solution().diagonalPrime([[1, 2, 3], [5, 17, 7], [9, 11, 10]]))  # 17

11
17


## 10. Vì sao thuật toán đúng

1. **Thu thập đủ ô.** Với mỗi `i` từ `0` đến `n - 1`, ta xét `nums[i][i]` (đường chéo chính) và `nums[i][n - 1 - i]` (đường chéo phụ). Hai tập chỉ số này đúng bằng định nghĩa "nằm trên đường chéo" của đề. Ô tâm (khi `n` lẻ) được xét hai lần, không sao vì ta chỉ lấy **max**.
2. **`is_prime` đúng.** Nếu `v < 2` thì không nguyên tố. Nếu `v >= 2` là hợp số `v = a × b` với `2 <= a <= b` thì `a <= √v`, nên `a` bị vòng `d = 2 .. isqrt(v)` bắt được (`v % a == 0`). Nếu `v` nguyên tố thì không có `d` nào trong `2..√v` chia hết, vòng lặp chạy hết và trả `True`.
3. **Cắt tỉa an toàn.** Invariant: sau mỗi bước, `best` = số nguyên tố lớn nhất trong các ô đã xét (hoặc `0` nếu chưa có). Ô mới có `v <= best` không thể làm tăng max ⇒ bỏ qua không đổi đáp án. Ô có `v > best` được kiểm tra đầy đủ nên `best` vẫn đúng.
4. **Kết quả cuối.** Khi duyệt hết, `best` là nguyên tố lớn nhất trong mọi ô đường chéo; nếu không có thì `best = 0` đúng yêu cầu của đề.

## 11. Phân tích Time / Space Complexity

`n` = số hàng (và cột), `M` = giá trị lớn nhất (`<= 4×10^6`).

| Cách | Time | Space |
|---|---|---|
| 1. Chia thử đến `v - 1` | `O(n · M)` | `O(1)` |
| 2. Chia thử đến `√v` | `O(n · √M)` | `O(1)` |
| 3. Cắt tỉa theo `best` | `O(n · √M)` tệ nhất, thường ít hơn | `O(1)` |
| 4. Sắp xếp giảm dần + dừng sớm | `O(n log n + k · √M)`, `k` là số giá trị phải kiểm tra | `O(n)` |
| 5. Sàng Eratosthenes | `O(M log log M + n)` | `O(M)` |

Với `n = 300`, `M = 4×10^6`: Cách 2/3 khoảng `10^6` phép chia, Cách 5 khoảng vài triệu phép gạch — cả hai đều trong giới hạn. Cách 1 là hàng tỷ phép chia nên **không** dùng cho constraints thật.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Gộp tất cả cách vào một danh sách rồi cho chạy cùng bộ test. Các edge case quan trọng: ma trận `1×1`, ô duy nhất là `1`; chỉ có số nguyên tố ở ô **không** nằm trên đường chéo; ô tâm; số `2`; số chính phương nguyên tố bình phương `49`; số nguyên tố lớn gần `4×10^6`.

In [19]:
sols = [SolutionBrute(), SolutionSqrt(), SolutionPrune(), SolutionSortedEarly(), SolutionSieve(), Solution()]
cases = [
    ("ví dụ 1", [[1, 2, 3], [5, 6, 7], [9, 10, 11]], 11),
    ("ví dụ 2", [[1, 2, 3], [5, 17, 7], [9, 11, 10]], 17),
    ("1x1 là 1", [[1]], 0),
    ("1x1 là 2", [[2]], 2),
    ("1x1 là 49", [[49]], 0),
    ("2x2: mọi ô đều trên đường chéo", [[4, 7], [13, 6]], 13),
    ("2x2: 7 và 11 cùng chéo phụ", [[4, 7], [11, 6]], 11),
    ("chỉ có nguyên tố ở ngoài", [[4, 6, 8], [9, 4, 9], [8, 6, 4]], 0),
    ("tâm là nguyên tố", [[4, 6, 8], [9, 13, 9], [8, 6, 4]], 13),
    ("hợp số bình phương", [[49, 4], [4, 49]], 0),
]
for name, mat, exp in cases:
    for s in sols:
        got = s.diagonalPrime([row[:] for row in mat])
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 11
PASS ví dụ 2 -> 17
PASS 1x1 là 1 -> 0
PASS 1x1 là 2 -> 2
PASS 1x1 là 49 -> 0
PASS 2x2: mọi ô đều trên đường chéo -> 13
PASS 2x2: 7 và 11 cùng chéo phụ -> 11
PASS chỉ có nguyên tố ở ngoài -> 0
PASS tâm là nguyên tố -> 13
PASS hợp số bình phương -> 0


Kiểm tra thủ công `[[4, 7], [13, 6]]`: với `n = 2` mọi ô đều nằm trên một đường chéo. Chéo chính: `4, 6`; chéo phụ: `nums[0][1] = 7`, `nums[1][0] = 13`. Nguyên tố: `7` và `13` ⇒ lớn nhất `13`. Ma trận `2×2` là ví dụ tốt để nhớ: **không có ô nào ở ngoài hai đường chéo**.

In [20]:
print(Solution().diagonalPrime([[4, 7], [13, 6]]))        # 13: cả 7 và 13 đều nằm trên đường chéo phụ
print(Solution().diagonalPrime([[4, 7], [11, 6]]))        # 11
print(Solution().diagonalPrime([[4, 5, 8], [9, 4, 9], [11, 6, 4]]))   # chéo phụ: 8, 4, 11 -> 11

13
11
11


### 12.2. Ca "nguyên tố nằm ngoài đường chéo"

Một ma trận `3×3` có đường chéo là các ô `(0,0), (0,2), (1,1), (2,0), (2,2)`; các ô ngoài đường chéo là `(0,1), (1,0), (1,2), (2,1)`. Đặt số nguyên tố lớn ở các ô ngoài đường chéo, đường chéo toàn hợp số ⇒ đáp án `0`.

In [21]:
mat = [[4, 97, 6],
       [89, 8, 83],
       [9, 79, 10]]
for s in sols:
    assert s.diagonalPrime([row[:] for row in mat]) == 0
print("PASS: nguyên tố chỉ nằm ngoài đường chéo => 0")

# đặt một nguyên tố nhỏ vào tâm
mat[1][1] = 5
for s in sols:
    assert s.diagonalPrime([row[:] for row in mat]) == 5
print("PASS: tâm = 5 => 5")

# phản ví dụ cho lỗi phổ biến: nguyên tố ở đầu trái trên trong khi chéo phụ toàn hợp
mat2 = [[7, 4, 4], [4, 4, 4], [4, 4, 4]]
for s in sols:
    assert s.diagonalPrime([row[:] for row in mat2]) == 7
print("PASS: chéo chính có 7 => 7")

PASS: nguyên tố chỉ nằm ngoài đường chéo => 0
PASS: tâm = 5 => 5
PASS: chéo chính có 7 => 7


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh ma trận ngẫu nhiên đúng constraints (`1 <= n <= 300` — ta dùng `n` nhỏ để chạy nhanh với cách chậm, và giá trị nhỏ để Cách 1 chạy được), so sánh mọi cách với một **oracle** viết từ định nghĩa (đếm số ước).

In [22]:
import random
random.seed(2614)

def oracle(nums):
    n = len(nums)
    cells = {nums[i][i] for i in range(n)} | {nums[i][n - 1 - i] for i in range(n)}
    def prime_by_def(v):
        divisors = [d for d in range(1, v + 1) if v % d == 0]
        return v > 1 and divisors == [1, v]
    primes = [v for v in cells if prime_by_def(v)]
    return max(primes) if primes else 0

for _ in range(300):
    n = random.randint(1, 8)
    hi = random.choice([5, 30, 200, 2000])
    mat = [[random.randint(1, hi) for _ in range(n)] for _ in range(n)]
    exp = oracle(mat)
    for s in sols:
        got = s.diagonalPrime([row[:] for row in mat])
        assert got == exp, (type(s).__name__, mat, got, exp)
print("300 ma trận ngẫu nhiên: mọi cách đều khớp oracle")

300 ma trận ngẫu nhiên: mọi cách đều khớp oracle


### 12.4. Test hiệu năng với giá trị lớn

Dùng các số nguyên tố lớn (gần `4×10^6`). Cách 2–5 phải chạy nhanh. Ta không chạy Cách 1 ở đây (quá chậm theo phân tích).

In [23]:
import time
def is_prime_check(v):
    return is_prime(v)

big_prime = max(v for v in range(4_000_000, 3_999_000, -1) if is_prime(v))
print("nguyên tố lớn nhất <= 4,000,000:", big_prime)
n = 300
big = [[big_prime - ((i * n + j) % 7) * 2 for j in range(n)] for i in range(n)]
big[150][150] = big_prime
fast = [SolutionSqrt(), SolutionPrune(), SolutionSortedEarly(), SolutionSieve(), Solution()]
for s in fast:
    t = time.perf_counter()
    r = s.diagonalPrime(big)
    print(f"{type(s).__name__:20} -> {r}  ({time.perf_counter() - t:.3f}s)")
    assert r == big_prime

nguyên tố lớn nhất <= 4,000,000: 3999971
SolutionSqrt         -> 3999971  (0.024s)
SolutionPrune        -> 3999971  (0.000s)
SolutionSortedEarly  -> 3999971  (0.000s)


SolutionSieve        -> 3999971  (0.021s)
Solution             -> 3999971  (0.000s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi `1` là số nguyên tố

Nhiều bạn viết vòng chia thử rồi trả `True` nếu không tìm thấy ước. Với `v = 1` vòng `range(2, 2)` rỗng ⇒ trả `True` — sai.

In [24]:
import math

def sai_quen_so_1(v):
    for d in range(2, math.isqrt(v) + 1):
        if v % d == 0:
            return False
    return True          # v = 1 cũng trả True: SAI

print("sai :", sai_quen_so_1(1))     # True (sai)

def dung_so_1(v):
    if v < 2:
        return False
    for d in range(2, math.isqrt(v) + 1):
        if v % d == 0:
            return False
    return True

print("đúng:", dung_so_1(1))         # False

sai : True
đúng: False


**Vì sao sai:** theo định nghĩa số nguyên tố phải **lớn hơn 1**. Đề cho `nums[i][j] >= 1` nên `1` rất hay xuất hiện (ví dụ 1 và 2 của đề đều có `1` ở góc). Luôn chặn `v < 2` đầu hàm.

### Lỗi 2: Cận trên vòng chia thử thiếu căn nguyên (bỏ sót số chính phương)

In [25]:
def sai_can_tren(v):
    if v < 2:
        return False
    for d in range(2, math.isqrt(v)):     # thiếu + 1: không thử d = isqrt(v)
        if v % d == 0:
            return False
    return True

print("sai :", sai_can_tren(9), sai_can_tren(25), sai_can_tren(49))   # True True True (sai hết)
print("đúng:", dung_so_1(9), dung_so_1(25), dung_so_1(49))             # False False False

sai : True True True
đúng: False False False


**Vì sao sai:** `range(a, b)` **không** chứa `b`. Với `v = 9`, `isqrt(9) = 3`; `range(2, 3)` chỉ có `2`, không thử `3` nên bỏ sót ước `3`. Cận trên phải là `isqrt(v) + 1`.

### Lỗi 3: Dùng sai chỉ số đường chéo phụ

In [26]:
from typing import List

def sai_cheo_phu(nums: List[List[int]]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        for v in (nums[i][i], nums[i][n - i]):        # n - i: vượt biên khi i = 0
            if dung_so_1(v) and v > best:
                best = v
    return best

try:
    sai_cheo_phu([[1, 2, 3], [5, 17, 7], [9, 11, 10]])
except IndexError as e:
    print("lỗi:", e)

def sai_cheo_phu_2(nums: List[List[int]]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        for v in (nums[i][i], nums[i][n - 1 - i]):
            if dung_so_1(v) and v > best:
                best = v
    return best
print(sai_cheo_phu_2([[1, 2, 3], [5, 17, 7], [9, 11, 10]]))   # đúng: 17

lỗi: list index out of range
17


**Vì sao sai:** cột của đường chéo phụ ở hàng `i` là `n - 1 - i` (hàng + cột = `n - 1`), **không phải** `n - i`. Cột `n` không tồn tại (cột hợp lệ là `0..n-1`). Mẹo kiểm: với `i = 0` cột phải là cột cuối `n - 1`.

### Lỗi 4: Chỉ xét một đường chéo

Đề nói "nằm trên **ít nhất một** đường chéo" nghĩa là xét **cả hai** đường chéo. Chỉ xét chéo chính sẽ bỏ sót các số nguyên tố ở chéo phụ.

In [27]:
def sai_mot_cheo(nums):
    n = len(nums)
    best = 0
    for i in range(n):
        v = nums[i][i]                     # chỉ chéo chính
        if v > best and dung_so_1(v):
            best = v
    return best

m = [[1, 2], [3, 4]]
print("sai :", sai_mot_cheo(m))            # 0: chéo chính chỉ có 1 và 4
print("đúng:", Solution().diagonalPrime(m))  # 3: chéo phụ có 2 và 3

sai : 0
đúng: 3


**Vì sao sai:** `[[1,2],[3,4]]` có chéo chính `1, 4` (không nguyên tố) và chéo phụ `2, 3` (cả hai nguyên tố) ⇒ đáp án đúng là `3`, không phải `0`. Hãy luôn thử một ví dụ mà đáp án chỉ nằm ở chéo phụ.

In [28]:
print(Solution().diagonalPrime([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))   # chéo 1,5,9 và 3,5,7 -> 7

7


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có "số nguyên tố" + giá trị `<= 10^6..10^7` + ít lần kiểm tra ⇒ **chia thử đến `√v`**. Nhiều truy vấn trên miền nhỏ (`<= 10^7`) ⇒ **sàng**.
- "Lớn nhất / nhỏ nhất" thoả điều kiện ⇒ cân nhắc **duyệt theo thứ tự ưu tiên rồi dừng sớm** hoặc **cắt tỉa theo `best`**.
- Đường chéo: nhớ hai công thức `i == j` và `i + j == n - 1`. Một vòng lặp `for i in range(n)` là đủ, không cần hai vòng lồng.
- Luôn tự hỏi: `1` có phải nguyên tố không? `2` là nguyên tố chẵn duy nhất. Cận trên `range` có bao gồm căn nguyên không?
- `math.isqrt` (Python 3.8+) an toàn hơn `int(math.sqrt(v))` vì không dính sai số số thực với số rất lớn.

## 15. Biến thể của bài

1. **Sum of Diagonals / Matrix Diagonal Sum (LeetCode 1572):** cộng các ô đường chéo, ô tâm chỉ cộng một lần.
2. **Count Primes (LeetCode 204):** đếm số nguyên tố nhỏ hơn `n` — bài sàng kinh điển.
3. **Prime Number of Set Bits in Binary Representation (LeetCode 762):** kiểm tra nguyên tố cho các số nhỏ.
4. **Đường chéo mở rộng:** tất cả ô cùng "đường chéo" `i - j = const` hoặc `i + j = const` (ví dụ Diagonal Traverse, LeetCode 498).
5. **Số nguyên tố nhỏ nhất / đếm số nguyên tố khác nhau trên đường chéo** — chỉ đổi phép gộp cuối.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chia thử đến `v - 1` | Theo định nghĩa | `O(n · M)` | `O(1)` | Chỉ để hiểu định nghĩa, giá trị nhỏ |
| 2. Chia thử đến `√v` | Hợp số có ước `<= √v` | `O(n · √M)` | `O(1)` | Đủ nhanh cho đề, ngắn gọn |
| 3. Cắt tỉa theo `best` | Bỏ giá trị `<= best` | `O(n · √M)`, thường nhanh hơn | `O(1)` | **Nên nộp**: gọn, ít kiểm tra |
| 4. Sắp xếp giảm dần + dừng sớm | Lớn nhất đứng đầu | `O(n log n + k·√M)` | `O(n)` | Khi muốn dừng ngay ở nguyên tố đầu tiên |
| 5. Sàng Eratosthenes | Lập bảng nguyên tố một lần | `O(M log log M + n)` | `O(M)` | Rất nhiều giá trị cần kiểm tra |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán kết quả của từng biểu thức rồi mới kiểm tra: `dung_so_1(1)`, `dung_so_1(2)`, `dung_so_1(49)`, `dung_so_1(97)`, `math.isqrt(50)`.

**Bài 2 (tính tay).** Với `nums = [[4,9,6,5],[8,15,7,2],[12,11,10,13],[14,3,1,16]]`, liệt kê các giá trị trên chéo chính, chéo phụ, các số nguyên tố trong đó, và đáp án `diagonalPrime`.

**Bài 3 (điền TODO).** Hoàn thành hàm `is_prime_todo(v)` (kiểm tra nguyên tố bằng chia thử đến căn).

In [29]:
import math

def is_prime_todo(v):
    # TODO 1: nếu v < 2 thì không phải nguyên tố
    # TODO 2: duyệt d từ 2 đến isqrt(v) (bao gồm isqrt)
    # TODO 3: nếu v chia hết cho d thì return False
    # TODO 4: duyệt xong không thấy ước thì return True
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `diagonal_prime_sieve(nums)` dùng sàng Eratosthenes (chỉ dùng `bytearray` hoặc list) mà **không nhìn** Cách 5. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Viết `count_diagonal_primes(nums)` trả về **số lượng giá trị nguyên tố khác nhau** nằm trên hai đường chéo. Ví dụ `[[1,2,3],[5,17,7],[9,11,10]]` cho `2` (giá trị `3` và `17`). Thử thêm: viết `smallest_diagonal_prime(nums)` trả về nguyên tố **nhỏ nhất** (hoặc `0` nếu không có).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1` không phải nguyên tố; `2` thì có; `49 = 7 × 7`; `97` không chia hết cho `2, 3, 5, 7` (chỉ cần thử đến `isqrt(97) = 9`). Bài 2: chéo chính `nums[0][0], nums[1][1], nums[2][2], nums[3][3]`; chéo phụ `nums[0][3], nums[1][2], nums[2][1], nums[3][0]`.
- **Mức 2 (vừa):** Bài 3: `if v < 2: return False`; `for d in range(2, math.isqrt(v) + 1)`; `if v % d == 0: return False`; cuối hàm `return True`. Bài 4: `is_p = bytearray([1]) * (M + 1)`, đặt `is_p[0] = is_p[1] = 0` (cẩn thận `M = 1`), gạch `is_p[p*p::p]` cho mỗi `p` còn nguyên tố với `p*p <= M`.
- **Mức 3 (sâu):** Bài 5: gom giá trị đường chéo vào `set` rồi đếm phần tử `is_prime`; nhớ ô tâm bị lặp nên cần `set`. Với "nhỏ nhất", dùng `min` trên danh sách nguyên tố hoặc `0` nếu rỗng (`min(..., default=0)`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert dung_so_1(1) is False
assert dung_so_1(2) is True
assert dung_so_1(49) is False
assert dung_so_1(97) is True
assert math.isqrt(50) == 7

# Bài 2
m = [[4, 9, 6, 5], [8, 15, 7, 2], [12, 11, 10, 13], [14, 3, 1, 16]]
n = len(m)
chinh = [m[i][i] for i in range(n)]
phu = [m[i][n - 1 - i] for i in range(n)]
assert chinh == [4, 15, 10, 16] and phu == [5, 7, 11, 14]
assert [v for v in chinh + phu if dung_so_1(v)] == [5, 7, 11]
assert Solution().diagonalPrime(m) == 11

# Bài 3
def is_prime_todo_ans(v):
    if v < 2:
        return False
    for d in range(2, math.isqrt(v) + 1):
        if v % d == 0:
            return False
    return True

assert [x for x in range(30) if is_prime_todo_ans(x)] == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

# Bài 4
def diagonal_prime_sieve(nums):
    n = len(nums)
    diag = [nums[i][i] for i in range(n)] + [nums[i][n - 1 - i] for i in range(n)]
    M = max(diag)
    is_p = bytearray([1]) * (M + 1)
    is_p[0] = 0
    if M >= 1:
        is_p[1] = 0
    for p in range(2, math.isqrt(M) + 1):
        if is_p[p]:
            is_p[p * p::p] = bytes(len(is_p[p * p::p]))
    return max((v for v in diag if is_p[v]), default=0)

assert diagonal_prime_sieve([[1, 2, 3], [5, 6, 7], [9, 10, 11]]) == 11
assert diagonal_prime_sieve([[1, 2, 3], [5, 17, 7], [9, 11, 10]]) == 17
assert diagonal_prime_sieve([[1]]) == 0

# Bài 5
def diagonal_values(nums):
    n = len(nums)
    return {nums[i][i] for i in range(n)} | {nums[i][n - 1 - i] for i in range(n)}

def count_diagonal_primes(nums):
    return sum(1 for v in diagonal_values(nums) if dung_so_1(v))

def smallest_diagonal_prime(nums):
    return min((v for v in diagonal_values(nums) if dung_so_1(v)), default=0)

m2 = [[1, 2, 3], [5, 17, 7], [9, 11, 10]]
assert count_diagonal_primes(m2) == 2
assert smallest_diagonal_prime(m2) == 3
assert smallest_diagonal_prime([[1]]) == 0
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cắt tỉa theo `best` + chia thử đến căn)

```python
import math
from typing import List

class Solution:
    def diagonalPrime(self, nums: List[List[int]]) -> int:
        def is_prime(v: int) -> bool:
            if v < 2:
                return False
            for d in range(2, math.isqrt(v) + 1):
                if v % d == 0:
                    return False
            return True

        n = len(nums)
        best = 0
        for i in range(n):
            for v in (nums[i][i], nums[i][n - 1 - i]):
                if v > best and is_prime(v):
                    best = v
        return best
```

**Checklist**

- [ ] `is_prime(1)` trả `False`? `is_prime(2)` trả `True`?
- [ ] Cận trên của vòng chia thử là `isqrt(v) + 1` (bao gồm căn nguyên)?
- [ ] Đường chéo phụ dùng cột `n - 1 - i`, không phải `n - i`?
- [ ] Ô tâm (n lẻ) bị xét hai lần có ảnh hưởng gì không? (Không, vì chỉ lấy max.)
- [ ] Mặc định `best = 0` khi không có số nguyên tố?
- [ ] Đã thử `[[1]]`, `[[2]]`, ma trận chỉ có nguyên tố ngoài đường chéo, và số lớn gần `4×10^6`?
- [ ] Signature đúng `diagonalPrime(self, nums: List[List[int]]) -> int` và có `import math`, `from typing import List`?